# ReVoice — XTTS-v2 Voice Cloning API (runs on Google Colab)

Run each cell below in order. The last cell will print a public URL —
copy that URL into `COLAB_API_URL` in your backend's `app.py`.

**Remember:** every time this notebook restarts, you get a NEW url
(unless you have a paid ngrok plan), so you'll need to update it again.

## 1. Install dependencies (takes a few minutes the first time)

In [1]:
!pip install -q TTS fastapi uvicorn pyngrok nest-asyncio soundfile

^C


## 2. Load the XTTS-v2 model onto the GPU

In [ ]:
import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("XTTS-v2 loaded successfully.")

## 3. Set your ngrok auth token
Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

## 4. Define the API — this is what your backend will call

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import base64, io, tempfile, soundfile as sf

app = FastAPI()

class SynthRequest(BaseModel):
    text: str
    speaker_wav_base64: str   # the reference voice clip, base64-encoded
    language: str = "en"

@app.get("/")
def health_check():
    return {"status": "XTTS-v2 API is running"}

@app.post("/synthesize")
def synthesize(req: SynthRequest):
    # Step 1: decode the incoming reference audio and save it temporarily
    ref_bytes = base64.b64decode(req.speaker_wav_base64)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(ref_bytes)
        ref_path = tmp.name

    # Step 2: run XTTS-v2 — this is the actual voice cloning step
    wav = tts.tts(text=req.text, speaker_wav=ref_path, language=req.language)

    # Step 3: encode the generated audio back to base64 to send to the backend
    buf = io.BytesIO()
    sf.write(buf, wav, samplerate=24000, format="WAV")
    audio_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    return {"audio_base64": audio_b64}

## 5. Start the server and expose it publicly via ngrok

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("=" * 60)
print("COPY THIS URL into your backend's app.py (COLAB_API_URL):")
print(public_url)
print("=" * 60)

uvicorn.run(app, host="0.0.0.0", port=8000)